In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import pandas as pd
import seaborn as sns

In [2]:
input_dir = '../../results/02_snakemake_simulate_data/output'

In [3]:
method_list = ['nnSVG', 'Spanve',  'SPARK-X', 'SPARK', 'SpatialDE2',
               'SpatialDE', 'SOMDE', 'MoranI', 'scGCO', 'SpaGCN',
               'SpaGFT', 'GPcounts', 'Sepal', 'BOOST-GP']

In [4]:
def compute_correlation(df, method):
    # reverse the ranking of genes for SPARK, SPARK-X
    if method == 'SPARK-X':
        df['adjustedPval'] = - df['adjustedPval']
    elif method == 'SPARK':
        df['adjusted_pvalue'] = - df['adjusted_pvalue']
    elif method == 'scGCO':
        df['fdr'] = - df['fdr']
    elif method == 'SpaGCN':
        df['pvals_adj'] = - df['pvals_adj']
        
    if method == 'GPcounts':
        df['log_likelihood_ratio'] = df['log_likelihood_ratio'].fillna(0)
        
    if method == 'BOOST-GP':
        df['PPI'] = df['PPI'].fillna(0)
        
    groupby = df.groupby('gene', observed=True)
    
    if method == 'nnSVG':
        corr = groupby.apply(lambda x: x['prop_sv'].corr(x['spatial_var'], method='kendall'))
    elif method == 'Spanve':
        corr = groupby.apply(lambda x: x['ent'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SPARK-X':
        corr = groupby.apply(lambda x: x['adjustedPval'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SPARK':
        corr = groupby.apply(lambda x: x['adjusted_pvalue'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SpatialDE2':
        corr = groupby.apply(lambda x: x['FSV'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SpatialDE':
        corr = groupby.apply(lambda x: x['FSV'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SOMDE':
        corr = groupby.apply(lambda x: x['FSV'].corr(x['spatial_var'], method='kendall'))        
    elif method == 'MoranI':
        corr = groupby.apply(lambda x: x['I'].corr(x['spatial_var'], method='kendall'))
    elif method == 'scGCO':
        corr = groupby.apply(lambda x: x['fdr'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SpaGCN':
        corr = groupby.apply(lambda x: x['pvals_adj'].corr(x['spatial_var'], method='kendall'))
    elif method == 'SpaGFT':
        corr = groupby.apply(lambda x: x['gft_score'].corr(x['spatial_var'], method='kendall'))
    elif method == 'GPcounts':
        corr = groupby.apply(lambda x: x['log_likelihood_ratio'].corr(x['spatial_var'], method='kendall'))
    elif method == 'Sepal':
        corr = groupby.apply(lambda x: x['sepal_score'].corr(x['spatial_var'], method='kendall'))
    elif method == 'BOOST-GP':
        corr = groupby.apply(lambda x: x['PPI'].corr(x['spatial_var'], method='kendall'))
    else:
        print(f'unknown method: {method}')
         
    return pd.DataFrame(data=corr)

In [5]:
def get_correlations(dataset_list):
    df_list = []
    for dataset in dataset_list:
        for method in method_list:
            input_filename = f'{input_dir}/{method}/{dataset}.csv'

            if os.path.exists(input_filename):
                df = pd.read_csv(input_filename)
                df_corr = compute_correlation(df, method)
            else:
                print(f'cannot find {input_filename}!')
                continue
            
            df_corr['dataset'] = dataset
            df_corr['method'] = method
        
            df_list.append(df_corr)
            
    df = pd.concat(df_list)
    df.columns = ['correlation', 'dataset', 'method']
    df['correlation'] = df['correlation'].fillna(0)
    df = df.reset_index()
    
    return df

In [6]:
dataset_list = ['01_10x_Visium_mouse_brain',
                '02_10x_Visium_Human_Heart_MI',
                '03_10x_Visium_Human_Colorectal_Cancer',
                '04_10x_Visium_Human_Heart',
                '05_10x_Visium_Mouse_Embryo',
                '06_10x_Visium_Mouse_Olfactory_Bulb',
                '07_10x_Visium_Human_Breast_Cancer',
                '08_10x_Visium_Human_Lymph_Node',
                '09_10x_Visium_Human_Prostate',
                '10_10x_Visium_Human_Prostate_Cancer',
                '11_10x_Visium_Human_Cerebellum',
                '12_10x_Visium_Mouse_Kidney',
                '13_10x_Visium_Human_Lung_Cancer',
                '14_10x_Visium_Human_Brain_Cancer',
                '15_10x_Visium_Human_Kidney',
                '16_10x_Visium_Human_Intestine_Cancer',
                '17_10x_Visium_Human_Skin_Melanoma',
                '18_10x_Visium_Human_Cervical_Cancer',
                '19_10x_Visium_Human_Breast_Cancer',
                '20_10x_Visium_Human_Heart_MI',
                '21_slide_seqv2_Mouse_Olfactory_Bulb',
                '22_slide_seqv2_Mouse_Cortex',
                '23_slide_seqv2_Mouse_Cerebellum',
                '24_slide_seqv2_Mouse_Hippocampus',
                '25_slide_seqv2_Mouse_SomatosensoryCortex',
                '26_DBiT_seq_E10_brain',
                '27_DBiT_seq_E10_eye',
                '28_DBiT_seq_E10_whole',
                '29_DBiT_seq_E11_lower_body',
                '30_DBiT_seq_E11_FL_1L',
                '31_DBiT_seq_E11_FL_2L',
                '32_10x_Xenium_Human_Colon_Cancer',
                '33_10x_Xenium_Mouse_Brain',
                '34_MERFISH_Human_cortex',
                '35_MERFISH_Human_cortex',
                '36_MERFISH_Human_cortex',
                '37_MERFISH_Human_cortex',
                '38_MERFISH_Mouse_cortex',
                '39_seqFISH_Mouse_Organogenesis',
                '40_slide_tag_mouse_embryo',
                '41_slide_tag_human_cortex',
                '42_slide_tag_human_skin_melanoma',
                '43_slide_tag_human_tonsil',
                '44_stereo_seq_Drosophila_embryo_E7',
                '45_stereo_seq_Drosophila_embryo_E9.1',
                '46_stereo_seq_Drosophila_embryo_E6.3',
                '47_stereo_seq_Drosophila_embryo_E10.5',
                '48_stereo_seq_Drosophila_embryo_E5.6',
                '49_STARmap_Wang2018_2D_zstep10_0',
                '50_STARmap_Wang2018_2D_zstep15_0'
               ]

df = get_correlations(dataset_list)

# ranks = df.groupby("method")["correlation"].median().fillna(0).sort_values()[::-1].index
# ax = sns.boxplot(df, x='method', y='correlation', order = ranks)
# ax.tick_params(axis='x', rotation=60)
# sns.despine(trim=True)

#df.to_csv('../../results/03_eval_simulate_data/10x_Visium_correlation.csv')

cannot find ../../results/02_snakemake_simulate_data/output/GPcounts/05_10x_Visium_Mouse_Embryo.csv!
cannot find ../../results/02_snakemake_simulate_data/output/SpaGCN/21_slide_seqv2_Mouse_Olfactory_Bulb.csv!
cannot find ../../results/02_snakemake_simulate_data/output/SOMDE/22_slide_seqv2_Mouse_Cortex.csv!
cannot find ../../results/02_snakemake_simulate_data/output/SpaGCN/22_slide_seqv2_Mouse_Cortex.csv!
cannot find ../../results/02_snakemake_simulate_data/output/GPcounts/22_slide_seqv2_Mouse_Cortex.csv!
cannot find ../../results/02_snakemake_simulate_data/output/SpaGCN/23_slide_seqv2_Mouse_Cerebellum.csv!
cannot find ../../results/02_snakemake_simulate_data/output/GPcounts/23_slide_seqv2_Mouse_Cerebellum.csv!
cannot find ../../results/02_snakemake_simulate_data/output/BOOST-GP/23_slide_seqv2_Mouse_Cerebellum.csv!
cannot find ../../results/02_snakemake_simulate_data/output/SpaGCN/24_slide_seqv2_Mouse_Hippocampus.csv!
cannot find ../../results/02_snakemake_simulate_data/output/BOOST-GP/

In [7]:
df.head()

,gene,correlation,dataset,method
0,1500015O10Rik,0.942857,01_10x_Visium_mouse_brain,nnSVG
1,2900040C04Rik,0.914286,01_10x_Visium_mouse_brain,nnSVG
2,6330403K07Rik,0.933333,01_10x_Visium_mouse_brain,nnSVG
3,Agrp,0.838095,01_10x_Visium_mouse_brain,nnSVG
4,Agt,0.952381,01_10x_Visium_mouse_brain,nnSVG


In [8]:
df.to_csv('../../results/03_eval_simulate_data/03_compute_correlation_per_gene.csv', 
          index=False)